In [44]:
import pandas as pd

In [45]:
df = pd.read_csv('bike_sales_100k.csv')
df.head()

,Sale_ID,Date,Customer_ID,Bike_Model,Price,Quantity,Store_Location,Salesperson_ID,Payment_Method,Customer_Age,Customer_Gender
0,1,11-07-2022,9390,Cruiser,318.32,1,Philadelphia,589,Apple Pay,70,Female
1,2,03-05-2024,3374,Hybrid Bike,3093.47,4,Chicago,390,Apple Pay,37,Male
2,3,01-09-2022,2689,Folding Bike,4247.99,3,San Antonio,338,PayPal,59,Female
3,4,28-09-2022,3797,Mountain Bike,1722.01,3,San Antonio,352,Apple Pay,19,Male
4,5,05-01-2021,1633,BMX,3941.44,3,Philadelphia,580,PayPal,67,Female


In [46]:
unique_products = list(set(df['Bike_Model']))
print(unique_products)  

['Electric Bike', 'Road Bike', 'BMX', 'Cruiser', 'Mountain Bike', 'Hybrid Bike', 'Folding Bike']


In [47]:
print(df.head())

   Sale_ID        Date  Customer_ID     Bike_Model    Price  Quantity  \
0        1  11-07-2022         9390        Cruiser   318.32         1   
1        2  03-05-2024         3374    Hybrid Bike  3093.47         4   
2        3  01-09-2022         2689   Folding Bike  4247.99         3   
3        4  28-09-2022         3797  Mountain Bike  1722.01         3   
4        5  05-01-2021         1633            BMX  3941.44         3   

  Store_Location  Salesperson_ID Payment_Method  Customer_Age Customer_Gender  
0   Philadelphia             589      Apple Pay            70          Female  
1        Chicago             390      Apple Pay            37            Male  
2    San Antonio             338         PayPal            59          Female  
3    San Antonio             352      Apple Pay            19            Male  
4   Philadelphia             580         PayPal            67          Female  


In [48]:
df['revenue'] = df['Price'] * df['Quantity']
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Year'] = df['Date'].dt.year

In [49]:
#1. Which bike models are the top performers by revenue?
#Total revenue by model
df.groupby(['Bike_Model'])['revenue'].sum().sort_values(ascending=False).head(7).reset_index()

,Bike_Model,revenue
0,Hybrid Bike,1.125055e+08
1,BMX,1.121461e+08
2,Cruiser,1.118491e+08
3,Road Bike,1.115796e+08
4,Folding Bike,1.109668e+08
5,Electric Bike,1.097502e+08
6,Mountain Bike,1.096370e+08


In [50]:
#Units sold per model
df.groupby(['Bike_Model'])['Quantity'].sum().sort_values(ascending=False).head(7).reset_index()

,Bike_Model,Quantity
0,Cruiser,43120
1,Hybrid Bike,43089
2,BMX,43080
3,Road Bike,43022
4,Folding Bike,42872
5,Mountain Bike,42279
6,Electric Bike,42249


In [55]:
#2. What are the monthly/quarterly sales trends?
#Monthly revenue trends
df['Month'] = df['Date'].dt.month
monthly_revenue = (
    df
    .groupby(['Year', 'Month'])['revenue']
    .sum()
    .reset_index()
    .pivot(index='Year', columns='Month', values='revenue')
    .sort_index()
)
monthly_revenue.head()

Month,1,2,3,4,5,6,7,8,9,10,11,12
Year,,,,,,,,,,,,
2020,13730133.97,13263040.47,13798941.14,13731412.04,13346550.96,13466750.65,13740939.33,13740345.86,13668682.31,14425089.02,13106503.20,14206101.55
2021,13397623.06,12098312.31,14328830.28,13399839.19,13899711.59,13286012.11,13556690.15,14410075.80,13502319.71,14109463.10,13785418.86,13760948.50
2022,13285233.34,12760647.09,14610155.87,13882613.20,13508660.29,13706309.36,14020423.25,13686964.73,13663398.39,14085581.15,13937161.98,14413600.84
2023,14052961.49,12556570.51,13684628.45,13126992.07,14251098.24,13804335.12,14203105.29,14315232.82,13958532.23,13015407.23,13670752.88,13857957.24
2024,14444749.13,12920346.12,13326290.26,13532725.53,14884378.63,13307413.12,14242926.94,13849983.44,10107363.56,NaN,NaN,NaN


In [52]:
#Quarterly revenue trends
df['Quarter'] = df['Date'].dt.quarter
Quarterly_revenue = (
    df
    .groupby(['Year', 'Quarter'])['revenue']
    .sum()
    .reset_index()
    .pivot(index='Year', columns='Quarter', values='revenue')
    .sort_index()
)
Quarterly_revenue.head()

Quarter,1,2,3,4
Year,,,,
2020,40792115.58,40544713.65,41149967.50,41737693.77
2021,39824765.65,40585562.89,41469085.66,41655830.46
2022,40656036.30,41097582.85,41370786.37,42436343.97
2023,40294160.45,41182425.43,42476870.34,40544117.35
2024,40691385.51,41724517.28,38200273.94,NaN


In [53]:
#Sales volume over time
VolumeOverTime = (
    df
    .groupby(['Year', 'Month'])['Quantity']
    .sum()
    .reset_index()
    .pivot(index='Year', columns='Month', values='Quantity')
    .sort_index()
    .fillna(0).astype(int)
)
VolumeOverTime.head()

Month,1,2,3,4,5,6,7,8,9,10,11,12
Year,,,,,,,,,,,,
2020,5352,5115,5378,5125,5281,5169,5384,5424,5211,5502,5164,5358
2021,5312,4661,5484,5045,5343,5077,5400,5342,5240,5479,5197,5363
2022,5193,4958,5570,5279,5235,5196,5364,5409,5280,5315,5204,5477
2023,5336,4803,5343,5117,5406,5382,5457,5466,5304,5119,5210,5448
2024,5500,4977,5103,5188,5678,5275,5331,5473,3909,0,0,0


In [61]:
#Year-over-year growth rates
YoY_growth = monthly_revenue.pct_change(axis=0) * 100
fix = monthly_revenue.notna() & monthly_revenue.shift(1).notna()
YoY_growth = YoY_growth.where(fix)
YoY_growth= YoY_growth.round(2)
YoY_growth = YoY_growth.dropna(how='all')
YoY_growth.head()



C:\Users\nugus\AppData\Local\Temp\ipykernel_16084\868211316.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  YoY_growth = monthly_revenue.pct_change(axis=0) * 100


Month,1,2,3,4,5,6,7,8,9,10,11,12
Year,,,,,,,,,,,,
2021,-2.42,-8.78,3.84,-2.41,4.14,-1.34,-1.34,4.87,-1.22,-2.19,5.18,-3.13
2022,-0.84,5.47,1.96,3.60,-2.81,3.16,3.42,-5.02,1.19,-0.17,1.10,4.74
2023,5.78,-1.60,-6.33,-5.44,5.50,0.72,1.30,4.59,2.16,-7.60,-1.91,-3.85
2024,2.79,2.90,-2.62,3.09,4.44,-3.60,0.28,-3.25,-27.59,NaN,NaN,NaN
